# Step 1: Problem Understanding & Framing
## Business Case
In industrial settings, machinery operates under varying conditions, leading to different "states" (normal, high-wear, near-failure, etc.). Traditional predictive maintenance often relies on labeled failure data, which is expensive, rare, and imbalanced. The goal is to identify distinct operating states and anomalies in machinery telemetry without using failure labels during training. This enables early detection of risky behaviors, proactive maintenance, and optimization of schedules. This could result to reducing unplanned downtime and costs.

## Machine Learning Task
Unsupervised clustering can help group machine operational behaviors based on sensor telemetry (temperatures, speed, torque, tool wear, etc.). We will be able to see operating patters when we plot these clusters. Anything that falls out of those patterns, or forms a tiny cluster in its own, is often a sign of abnormal behavior or a developing issue. This is a typical approach in predictive maintenance to spot problems early.

## Technical Evaluation Metrics

* **Silhouette Score**: Measures how similar an instance is to its own cluster vs. others (range: -1 to 1; higher is better). Assesses cohesion and separation.
* **Davies-Bouldin Index**: Ratio of within-cluster scatter to between-cluster separation (lower is better).
* **Elbow Method (Inertia / Within-Cluster Sum of Squares - WCSS)**: Plots inertia vs. number of clusters (k) to find the "elbow" where additional clusters yield diminishing returns.

Additional diagnostics: Cluster stability, visualization of projections (PCA/t-SNE/UMAP), and post-hoc validation against held-out failure labels.

## Business KPIs

1. **Reduction in Unplanned Downtime**: Percentage decrease in unexpected failures by flagging "at-risk" clusters early (target: 15-30% reduction via timely interventions).
2. **Maintenance Schedule Optimization**: Increase in predictive (vs. reactive) maintenance actions and reduction in unnecessary preventive replacements (e.g., tool changes based on cluster-derived wear states).
3. **Overall Equipment Effectiveness (OEE) Improvement**: Measured via availability, performance, and quality uplift from better state awareness.

These KPIs tie directly to ROI through lower costs and higher throughput.

# Step 2: Data Collection & Understanding
## Dataset Overview
The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset (modeled on real industrial milling processes) with 10,000 instances and 14 columns. It includes process parameters and failure indicators. For strict unsupervised clustering, we will drop or separate all target/failure columns (Machine failure, TWF, HDF, PWF, OSF, RNF) during modeling. These will only be used post-hoc for cluster validation/profiling.
Key telemetry features capture machine operating conditions under different product quality variants.

Source: https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020

## Data Dictionary
| Feature            | Type        | Unit | Description / Operational Range                     | Role              |
|--------------------|-------------|------|------------------------------------------------------|-------------------|
| UDI                | Integer     | -    | Unique identifier (1–10000)                          | Drop (ID)         |
| Product ID         | Categorical | -    | L/M/H + serial number                                | Drop (ID)         |
| Type               | Categorical | -    | Product quality variant: L (50%), M (30%), H (20%)   | Feature (encode)  |
| Air temperature    | Continuous  | K    | ~300K ± 2K (random walk)                             | Feature           |
| Process temperature| Continuous  | K    | Air temp + ~10K ± 1K                                 | Feature           |
| Rotational speed   | Integer     | rpm  | Derived from ~2860W power, with noise                | Feature           |
| Torque             | Continuous  | Nm   | ~40 Nm ± 10 Nm (no negatives)                        | Feature           |
| Tool wear          | Integer     | min  | Cumulative wear; varies by Type (H/M/L add extra wear)| Feature          |
| Machine failure    | Binary      | -    | 0/1 (any failure)                                    | Drop for training |
| TWF / HDF / PWF / OSF / RNF | Binary | - | Specific failure modes                               | Drop for training |

## Data Undersrtanding

| Feature | Meaning | Why it matters for failure |
|---|---|---|
| Air temperature [K] | Ambient temp | High temp → overheating risk |
| Process temperature [K] | Internal temp | Strong indicator of thermal stress |
| Rotational speed [rpm] | Motor speed | Abnormal speed → mechanical wear |
| Torque [Nm] | Load on machine | High torque → strain → failure |
| Tool wear [min] | Usage time | Direct proxy for degradation |
| Type | Product type | Different products → different stress patterns |

**Notes:**
* Total ~3.4% failure rate (highly imbalanced, as expected in real PM data).
* No missing values.
* Features have different scales/magnitudes → scaling is critical for distance-based clustering.


# Step 3: Data Preprocessing, Applied EDA & Feature Engineering
## 3.1 Data Cleaning

* Remove non-predictive attributes (UDI, Product ID).
* Separate failure labels (Machine failure and failure mode columns). They are not needed in the unsupervised model training. They will be used for validation only.
* Check for missing values/duplicates (dataset is clean, but code includes guards).

* Handle the categorical values in the 'Type' column (L/M/H).

**Null values and Column Types**
![alt text](../presentation/images/eda_nulls_and_coltypes.png)

**Variable Summary**
![alt text](../presentation/images/eda_variable_summary.png)

**Selected Features Distribution**
![alt text](../presentation/images/eda_selected_feature_dist.png)

## 3.2 Applied EDA

* Assess clustering tendency (Hopkins statistic or visual inspection).
![alt text](../presentation/images/eda_clustering_tendency_hopkins.png)
* Univariate distributions and pairwise correlations (distance-based algorithms are sensitive to scale and multicollinearity).
* Domain context: Temperatures are correlated; power-related features (speed × torque) may be informative.

## 3.3 Feature Engineering & Scaling

* Scaling: Mandatory for K-Means, DBSCAN, Hierarchical (Euclidean distance). Use StandardScaler (robust to outliers in telemetry).
* Encoding: One-hot or ordinal for Type (quality affects wear).
* Derived Features:
  * temp_diff = Process temperature - Air temperature (heat dissipation proxy).
  * power_proxy = Torque * Rotational speed (approximates mechanical power/load).

* Feature Selection: Remove highly correlated features or low-variance ones to reduce noise.

## 3.4 Dimensionality Reduction

* PCA: Reduces dimensions while preserving variance; improves distance-based clustering and mitigates curse of dimensionality.
* t-SNE / UMAP: Visualization only (not for modeling) to inspect cluster separability in 2D.
![alt text](../presentation/images/eda_tsne.png)

Interpretation: 
**Failure Lablel**: 
* There is an absence of a distinct failure zone. Failure points overlaps heavily with normal operation.
* Machine Failures occur within the same operating space as normally operating machines.
* Unsupervised clustering alone is not enough to detect failures. Supervised learning is also needed. 

**Product Tye**
* Product types exhibit mild operational differences, but their t‑SNE projections show overlap, indicating that product type alone does not define distinct machine states

**Torque**
* Shows machine states into distinct regions, suggesting that torque is a key contributor to operational differences and potentially to failure risk i.e., High torque = high load = mechanical stress
* Stress increases wear and failure probability